<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Theoretical Foundations

This notebook contains the theory corresponding to the implementation stages in `Background_Subtraction.ipynb`. The implementation notebook remains code/output-focused; mathematical justification, assumptions, metric definitions, and interpretation live here.

### Core model
For an observed frame $I_t$ and processed background reference $B$, the signed residual used by the implementation is based on

$$D_t = -(I_t-B).$$

The residual is transformed through spatial filtering, radiometric normalization, Fourier-domain high-pass filtering, morphology, and segmentation. The final mask convention is `0 = tool`, `1 = background`.

## 1. Configure the Processing Environment

A reproducible image-processing experiment starts by fixing all inputs and hyperparameters before evaluation. Repository-relative paths prevent dependence on a specific machine, while named constants make the experiment auditable.

The frame set is $\{201,211,\ldots,291\}$, frame 201 is the static background reference, and frames 211–291 form the evaluation sequence. Centralizing Gaussian scale, percentile limits, spectral cutoff, morphology radii, threshold, and GMM configuration ensures that later comparisons differ only where intended.

## 2. Define Reusable Processing and Evaluation Functions

The implementation decomposes the workflow into reusable operators. This enforces the same mathematics for every frame and every segmentation strategy.

Percentile normalization maps an image to $[0,1]$ using lower and upper limits $L$ and $H$:

$$I'(x,y)=\operatorname{clip}\left(\frac{I(x,y)-L}{H-L+\varepsilon},0,1\right).$$

Ground truth is the union of the guidewire and microcatheter annotations. Reusable functions also isolate spatial smoothing, spectral filtering, segmentation, morphology, metrics, and visualization so downstream comparisons remain controlled.

## 3. Validate the Dataset and Annotations

Quantitative evaluation is only meaningful if every image is paired with the correct annotations and all arrays are spatially compatible. The dataset-validation cell therefore checks file existence and enforces one common image shape.

These assertions are part of the experimental method: a numerically valid metric computed from a mismatched frame or mask would still be scientifically invalid.

## 4. Build the Static Background Reference

The project uses frame 201 as a fixed reference. Let the preprocessing operator be $P(\cdot)$; the background is

$$B=P(I_{201}).$$

Every evaluation frame is transformed through the same preprocessing operator before subtraction. Using a single processed background makes temporal differences interpretable and keeps all segmentation strategies downstream of the same residual.

## 5. Inspect the Intermediate Pipeline on a Representative Frame

Frame 251 is used to expose the complete feature-construction chain. Gaussian spatial smoothing suppresses small high-frequency fluctuations:

$$G_\sigma(x,y)=\frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right).$$

After preprocessing and signed background subtraction, the residual is transformed to the Fourier domain. The diagnostic views make it possible to inspect whether each processing stage preserves the thin moving-tool response while suppressing static anatomical structure.

## 6. Generate the Three Candidate Segmentation Masks

All three segmentation methods operate on the same feature image.

**Fixed threshold:**
$$M_{\text{tool}}(x,y)=\mathbf{1}[R(x,y)>T].$$

**Otsu:** the threshold is selected automatically by maximizing between-class variance in the scalar feature distribution.

**EM/GMM:** a two-component Gaussian mixture models the feature values as
$$p(x)=\sum_{k=1}^{2}\pi_k\,\mathcal{N}(x\mid\mu_k,\Sigma_k).$$
The component with the larger mean response is treated as tool-like.

The comparison is fair because only the segmentation decision rule changes.

## 7. Compare the Candidate Strategies Qualitatively

A binary mask alone does not reveal error semantics. The validation overlay separates three cases: correct overlap, missed ground-truth tool pixels, and false-positive detections.

This visual comparison complements scalar metrics by showing *where* each method fails, which matters for thin structures such as guidewires and microcatheters.

## 8. Evaluate All Strategies Across the Sequence

Every candidate strategy is evaluated on the same frames and with the same upstream feature construction. The implementation reports both laboratory-style pixel errors and overlap metrics.

For $N$ pixels,

$$\mathrm{SAD}=\frac{1}{N}\sum_{i=1}^{N}|G_i-M_i|,$$

$$\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(G_i-M_i)^2,$$

$$\mathrm{PSNR}=20\log_{10}\left(\frac{L}{\sqrt{\mathrm{MSE}}}\right).$$

For tool-positive binary supports, Dice and IoU are

$$\mathrm{Dice}=\frac{2|A\cap B|}{|A|+|B|},\qquad \mathrm{IoU}=\frac{|A\cap B|}{|A\cup B|}.$$

## 9. Aggregate Metrics and Select the Retained Strategy

Per-frame measurements are summarized by mean and standard deviation for each strategy. The implementation sorts strategies by mean MSE and retains the one with the lowest mean MSE.

The important methodological point is that the selection rule is explicit and reproducible. The retained method is not chosen from a single visually favorable frame.

## 10. Plot Temporal Validation Curves

A sequence mean can hide unstable behavior. Plotting SAD, MSE, and PSNR as functions of frame number reveals temporal variation:

$$m_t=f(t),$$

where $m_t$ is the metric at frame $t$. Large spikes identify frames for which the background-subtraction assumptions or segmentation rule fail locally.

## 11. Summarize Strategy-Level Performance

Compact bar summaries of mean MSE, mean PSNR, and mean Dice provide complementary views of performance. Error metrics reward pixel-wise agreement, while Dice emphasizes overlap of the sparse tool class.

Because these metrics measure different properties, the summary is interpreted together with the temporal curves and qualitative overlays rather than in isolation.

## 12. Assemble the Final Retained Pipeline

The final workflow encapsulates the experimentally selected strategy behind one interface:

$$I_t \rightarrow P(I_t) \rightarrow D_t \rightarrow H_{\mathrm{HP}} \rightarrow \text{morphology} \rightarrow \text{selected segmentation} \rightarrow M_t.$$

The Gaussian high-pass transfer function used by the implementation is

$$H_{\mathrm{HP}}(u,v)=1-\exp\left(-\frac{D(u,v)^2}{2D_0^2}\right).$$

Encapsulation ensures that all later presentation and validation use the same retained pipeline.

## 13. Generate the Final Guidance Gallery

The final prediction is overlaid on a contrast-enhanced fluoroscopic image to produce a guidance view. This is a presentation layer only: it does not change the segmentation result.

The overlay is useful because a clinically meaningful tool detector must be interpretable in the context of the underlying anatomy, not only as an isolated binary mask.

## 14. Perform Final Qualitative Validation

Representative early, middle, and late frames are compared using prediction, manual annotation, and semantic error overlay. This samples the sequence temporally and exposes whether a method that performs well on average degrades at particular stages.

Qualitative validation is therefore a guardrail against over-reliance on aggregate metrics.

## 15. Export Results and Verify Deliverables

The final step preserves the complete numerical evidence in CSV files and confirms that the diagnostic figures were generated. Reproducibility requires both computations and durable artifacts.

A complete run therefore produces: per-frame metrics, strategy-level summary statistics, representative diagnostics, temporal curves, final guidance views, and final qualitative validation figures.

## Technical Synthesis

The project follows a controlled experimental sequence:

$$\boxed{\text{configuration}\rightarrow\text{reusable operators}\rightarrow\text{data validation}\rightarrow\text{background model}\rightarrow\text{feature construction}\rightarrow\text{strategy comparison}\rightarrow\text{sequence evaluation}\rightarrow\text{strategy selection}\rightarrow\text{final pipeline}\rightarrow\text{export}}$$

The numbering above intentionally maps **1:1 to the 15 code cells** in the implementation notebook. If the executable structure changes, the theory notebook must be updated to match rather than forced into a fixed task count.

## Scope and Limitations

The implemented method assumes that most anatomical content remains sufficiently stable relative to the moving tools. It does not include non-rigid registration, optical flow, learned segmentation, adaptive online background models, uncertainty calibration, or clinical validation. Performance can degrade when anatomy moves substantially, intensity statistics drift, or the foreground tool response is not separable from residual background structure.